# FEATURES_NO_LEAK (for predicting cluster_1)

In [ ]:
# ============================================================
# FEATURES_NO_LEAK (for predicting cluster_1)
# Goal: explain what provider characteristics correlate with being in cluster_1
# Avoid: using the anomaly/tiering features that were used to build clusters (leakage)
# ============================================================

# Provider covariates we will aggregate from eval_base (eval_scored_DG_V3.parquet)
FEATURES_NO_LEAK_NUM = [
    # case-mix / comorbidity prevalence (provider-level, repeated per row; safe to aggregate)
    "p_cancer6",
    "p_diabetes",
    "p_ckd",
    "p_copd",
    "p_htn",

    # risk / provider maturity (also safe)
    "bene_avg_risk_score",
    "years_since_enumeration",
]

FEATURES_NO_LEAK_CAT = [
    # geography / practice context
    "ruca_bucket",     # categorical bucket is nice for interpretability
]

# These are not "leakage", but they are operationally tied to our eligibility gate
# and correlate strongly with opportunity to accumulate events.
# Use them only if we want a "what predicts being on the watchlist, including scale" story.
FEATURES_OPTIONAL_SCALE = [
    "total_services_base",
    "total_benes_base",
    "log_total_services_base",
    "log_total_benes_base",
]

# Categorical keys already in provider_scorecard_v1_clustered (we will one-hot encode)
# - provider_type
# - state

# CLF.0) Load clustered provider scorecard + eval_base (single source of truth)

In [ ]:
# ============================================================
# CLF.0) Load clustered provider scorecard + eval_base (single source of truth)
# Defines:
#   - scorecard (provider grain, labeled with cluster)
#   - eval_base (row grain)
#   - eligible_scorecard (only clustered providers)
#   - y (target: is_cluster_1)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ---- Paths (edit only if our run timestamp changes) ----
SCORECARD_PATH = Path("artifacts/provider_tiering/run_20260317_085502/provider_scorecard_v1_clustered__20260317_085502.parquet")
EVAL_PATH = Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")

if not SCORECARD_PATH.exists():
    raise FileNotFoundError(f"Missing SCORECARD_PATH: {SCORECARD_PATH}")
if not EVAL_PATH.exists():
    raise FileNotFoundError(f"Missing EVAL_PATH: {EVAL_PATH}")

scorecard = pd.read_parquet(SCORECARD_PATH)
eval_base = pd.read_parquet(EVAL_PATH)

print("Loaded scorecard:", scorecard.shape)
print("Loaded eval_base:", eval_base.shape)

KEY = ["Rndrng_NPI", "provider_type", "state"]
for c in KEY + ["cluster_label_v1"]:
    if c not in scorecard.columns:
        raise KeyError(f"scorecard missing required col: {c}")
for c in KEY:
    if c not in eval_base.columns:
        raise KeyError(f"eval_base missing required key col: {c}")

# ---- Key uniqueness sanity ----
dup = scorecard.duplicated(KEY).sum()
if dup != 0:
    raise ValueError(f"Scorecard has duplicate provider keys: {dup}")

# ---- Eligible subset for classification ----
eligible_scorecard = scorecard[scorecard["cluster_label_v1"].astype(str).str.startswith("cluster_")].copy()
print("Eligible providers (clustered):", len(eligible_scorecard))
print("Cluster counts:\n", eligible_scorecard["cluster_label_v1"].value_counts())

# Target
eligible_scorecard["is_cluster_1"] = (eligible_scorecard["cluster_label_v1"] == "cluster_1").astype(int)
y = eligible_scorecard["is_cluster_1"].to_numpy()
print("Target prevalence (cluster_1 %):", float(y.mean() * 100))

# CLF.1) Aggregate eval_base -> provider grain, then merge onto eligible_scorecard

In [ ]:
# ============================================================
# CLF.1) Aggregate eval_base -> provider grain, then merge onto eligible_scorecard
# Produces:
#   - cov (provider-level covariates from eval_base)
#   - clf_df (eligible_scorecard + covariates)
# Notes:
#   - Weighted averages use benes when available.
#   - ruca_bucket is taken as mode (most frequent) per provider.
# ============================================================

import pandas as pd
import numpy as np

# Columns we want from eval_base
FEATURES_NO_LEAK_NUM = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
FEATURES_NO_LEAK_CAT = ["ruca_bucket"]

need_cols = KEY + FEATURES_NO_LEAK_NUM + FEATURES_NO_LEAK_CAT + ["benes", "services"]
missing = [c for c in need_cols if c not in eval_base.columns]
if missing:
    raise KeyError(f"eval_base missing required cols for aggregation: {missing}")

df = eval_base[need_cols].copy()

# Ensure numerics
for c in FEATURES_NO_LEAK_NUM + ["benes", "services"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Helper: weighted mean (fallback to simple mean if weights degenerate)
def _wmean(x: pd.Series, w: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce").fillna(0)
    mask = x.notna() & np.isfinite(x) & np.isfinite(w)
    if mask.sum() == 0:
        return np.nan
    x = x[mask]
    w = w[mask]
    sw = float(w.sum())
    if sw <= 0:
        return float(x.mean())
    return float((x * w).sum() / sw)

def _mode_str(x: pd.Series) -> str:
    x = x.astype(str)
    vc = x.value_counts(dropna=False)
    return str(vc.index[0]) if len(vc) else "nan"

# Aggregate to provider grain
g = df.groupby(KEY, dropna=False)

cov = g.apply(lambda d: pd.Series({
    # case-mix (often constant per provider across rows, but safe to aggregate)
    "p_cancer6": _wmean(d["p_cancer6"], d["benes"]),
    "p_diabetes": _wmean(d["p_diabetes"], d["benes"]),
    "p_ckd": _wmean(d["p_ckd"], d["benes"]),
    "p_copd": _wmean(d["p_copd"], d["benes"]),
    "p_htn": _wmean(d["p_htn"], d["benes"]),
    # risk / maturity
    "bene_avg_risk_score": _wmean(d["bene_avg_risk_score"], d["benes"]),
    "years_since_enumeration": float(np.nanmedian(d["years_since_enumeration"].to_numpy())),
    # context
    "ruca_bucket": _mode_str(d["ruca_bucket"]),
    # optional scale (explicitly named as "base" so it is clear it comes from eval_base)
    "total_services_base": float(np.nansum(d["services"].to_numpy())),
    "total_benes_base": float(np.nansum(d["benes"].to_numpy())),
}), include_groups=False).reset_index()

# Add optional scale logs (safe transform)
cov["log_total_services_base"] = np.log1p(pd.to_numeric(cov["total_services_base"], errors="coerce").fillna(0))
cov["log_total_benes_base"] = np.log1p(pd.to_numeric(cov["total_benes_base"], errors="coerce").fillna(0))

print("Aggregated covariates:", cov.shape)

# Merge onto eligible scorecard
clf_df = eligible_scorecard.merge(cov, on=KEY, how="left")

# Merge sanity
n_missing_cov = int(clf_df["bene_avg_risk_score"].isna().sum())
print("Eligible providers:", len(clf_df))
print("Providers missing covariates after merge:", n_missing_cov)

# If anything is missing (should be rare), fill with medians / 'unknown' so models can run
for c in FEATURES_NO_LEAK_NUM + ["total_services_base","total_benes_base","log_total_services_base","log_total_benes_base"]:
    if c in clf_df.columns:
        clf_df[c] = pd.to_numeric(clf_df[c], errors="coerce")
        clf_df[c] = clf_df[c].fillna(float(clf_df[c].median()))
clf_df["ruca_bucket"] = clf_df["ruca_bucket"].astype(str).fillna("unknown")

print("CLF frame ready:", clf_df.shape)

# CLF.2) Train baseline classifier (eligible only) + interpret drivers

In [ ]:
# ============================================================
# CLF.2) Train baseline classifier (eligible only) + interpret drivers
# Model:
#   - Logistic Regression (balanced) inside a preprocessing pipeline
# Outputs:
#   - ROC AUC, PR AUC
#   - Confusion at 0.5 threshold
#   - Top positive/negative coefficients (interpretation)
# Notes:
#   - Start with FEATURES_NO_LEAK_* only.
#   - We can optionally add scale features to see how much "size" explains cluster_1.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

# Base features (no-leak)
NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type", "state"]

# Optional scale features (toggle)
USE_SCALE = False
if USE_SCALE:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base", "log_total_benes_base"]

# Build X/y
X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

# Model
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

# Predict
p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SCALE =", USE_SCALE)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

# ---- Interpret coefficients (log-odds) ----
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs}).sort_values("coef_log_odds", ascending=False)

print("\nTop positive drivers (higher => more likely cluster_1):")
display(coef_df.head(20))

print("\nTop negative drivers (lower => less likely cluster_1):")
display(coef_df.tail(20).iloc[::-1])

## Inspect all the coefficients of logistic regression model

In [ ]:
with pd.option_context("display.max_rows", None):
    display(coef_df)

## Evaluate credibility of the coefficients based on sample size

In [ ]:
tmp = clf_df.groupby("state", as_index=False)["is_cluster_1"].value_counts()

with pd.option_context("display.max_rows", None):
    display(tmp)

# CLF.2.a) Train baseline classifier (eligible only) + interpret drivers (ADDS SCALE INDICATORS SERVICE VOLUME AND BENEFICIARY COUNTS)

In [ ]:
# ============================================================
# CLF.2.a) Train baseline classifier (eligible only) + interpret drivers (ADDS SCALE INDICATORS SERVICE VOLUME AND BENEFICIARY COUNTS)
# Model:
#   - Logistic Regression (balanced) inside a preprocessing pipeline
# Outputs:
#   - ROC AUC, PR AUC
#   - Confusion at 0.5 threshold
#   - Top positive/negative coefficients (interpretation)
# Notes:
#   - Start with FEATURES_NO_LEAK_* only.
#   - We can optionally add scale features to see how much "size" explains cluster_1.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

# Base features (no-leak)
NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type", "state"]

# Optional scale features (toggle)
USE_SCALE = True
if USE_SCALE:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base", "log_total_benes_base"]

# Build X/y
X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

# Model
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

# Predict
p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SCALE =", USE_SCALE)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

# ---- Interpret coefficients (log-odds) ----
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs}).sort_values("coef_log_odds", ascending=False)

print("\nTop positive drivers (higher => more likely cluster_1):")
display(coef_df.head(20))

print("\nTop negative drivers (lower => less likely cluster_1):")
display(coef_df.tail(20).iloc[::-1])

### 1) What changed when we turned on scale features

Performance improved:
* **ROC AUC:** 0.714 → 0.736 (meaningful lift)
* **PR AUC:** 0.418 → 0.460 (also meaningful, and PR is the more relevant metric with 24% prevalence)
* **Accuracy:** 0.627 → 0.661
* **Class 1 (`cluster_1`) recall:** 0.748 → 0.705 (slightly down)
* **Class 1 precision:** 0.369 → 0.392 (slightly up)

So adding size is helping the rank-ordering (AUCs), and the operating point at 0.5 is trading recall for a bit more precision. That’s totally normal.

---

### 2) Interpreting the `log_total_services_base` coefficient

* `log_total_services_base` coef = 0.737
* **Odds multiplier per +1 SD:** exp(0.737) ≈ 2.09x

Because it’s standardized, this means:
* A provider with 1 standard deviation higher `log(total services)` has about 2x higher odds of being in `cluster_1`, holding other covariates fixed.

That is a strong signal, and it matches what we already saw in our tiering profiles: `cluster_1` providers are much larger on median services. 

Also important: it’s not “huge” relative to the strongest state coefficients, but it is one of the top drivers and it moved `provider_type` down the list. That’s the key evidence that size explains a real chunk of why providers end up in `cluster_1`.

---

### 3) Why `provider_type` got smaller once scale was added

In CLF.2 (no scale), `provider_type` was dominant (Medical Oncology ~1.21 log-odds).
In CLF.2.a (with scale):
* `provider_type_Medical Oncology` = 0.639 (exp ≈ 1.90x odds)
* `provider_type_Hematology-Oncology` = 0.445 (exp ≈ 1.56x odds)

This pattern usually means:
* Some of what “looks like specialty risk” is actually “specialties that tend to be higher volume in this eligible set.”

So specialty still matters, but the model is now attributing part of that signal to volume, which is more honest.

---

### 4) Geography is still strong, but interpret it with support thresholds

States still show up heavily. That’s expected because state is a high-cardinality categorical and can capture:
* practice pattern differences
* billing mix differences
* data quirks / coverage differences
* residual geography effects even after standardization

But the same caution holds as before: **AK, WY, XX, PR** are likely small-n artifacts. Use a support threshold (like eligible providers in state ≥ 100) before we put any state callouts on a slide.

A good “slide-safe” way to talk about geography is:
> “After controlling for case-mix proxies, RUCA, specialty, and size, there remains geographic variation in `cluster_1` prevalence.”

Then show it aggregated (region) or show only high-support states.

---

### 5) What this result says about the meaning of `cluster_1`

This is the core takeaway:
* Our unsupervised clusters were intended to be value-focused (excluding volume from distance).
* Yet, even within the eligible set, `cluster_1` is correlated with size (services). The classifier confirms it.

That does not invalidate the tiering. It actually clarifies the product interpretation:
> *`cluster_1` is “high anomaly burden” and it tends to occur among higher-volume, higher-confidence providers, because they have more opportunity to accumulate robust tail events.*

That is a very defensible story.

#### Checking the full `coef_df` dataframe output

In [ ]:
with pd.option_context("display.max_rows", None):
    display(coef_df)

### 1) What the “scale” story looks like now

**`log_total_services_base` is a real driver**
* **coef** = 0.737
* Because it’s standardized, that’s “per +1 SD in log(total services)”
* **odds multiplier:** exp(0.737) ≈ 2.09x

So size, measured by services, strongly increases the odds of being `cluster_1`.

**`log_total_benes_base` is basically doing nothing (in this model)**
* **coef** = 0.012
* **odds multiplier:** exp(0.012) ≈ 1.01x per +1 SD

That’s effectively zero. The usual interpretation is:
1. services captures the size signal better than benes for our `cluster_1` label, or
2. benes is largely redundant with services, and once services is in, benes adds almost no incremental information.

Given our tiering outputs, this makes sense: `cluster_1` skewed very high on services. Benes rises too, but services is the stronger “exposure/opportunity” proxy for accumulating robust tail events.

If we want to make this airtight, do a tiny follow-up run:
* model with only `log_total_benes_base` (no services)
* model with only `log_total_services_base`
* model with both (current)

We'll likely see benes matter only when services is removed.

---

### 2) Provider type effects stayed, but got “deconfounded” by size

* **Medical Oncology:** 0.639 (≈ 1.90x odds per indicator)
* **Hematology-Oncology:** 0.445 (≈ 1.56x odds)
* **Radiation Oncology:** 0.260 (≈ 1.30x odds)
* **Surgical Oncology:** 1.215 (≈ 0.30x odds)

This is a good interpretability story:
> Specialty still matters, but once we account for size (services), the specialty effect shrinks. That suggests some of the earlier “specialty risk” signal was actually “specialties that tend to operate at higher service volume in this dataset.”

---

### 3) Geography is still dominating because one-hot categories can be loud

We have multiple states with coefficients larger than most clinical covariates (and some larger than `provider_type`). Two points to keep our interpretation honest:

**A) Small-n states will look extreme** AK, WY, PR, VI, XX are classic “don’t put on slides without support” states. They can be real, but they can also be noise from tiny denominators. 

A clean practice is:
* only interpret state coefficients for states with, say, **>= 100 eligible providers** (or whatever threshold we choose)
* otherwise treat them as “unstable estimates”

**B) States can be proxying for unmodeled factors** Even with standardized cost, state can still proxy:
* practice patterns
* distribution of provider types within the state
* coding mix not captured by our chosen covariates
* residual geography effects

So it’s totally fine that state is strong. We just want to treat it as **context**, not as an “explanation” we over-emphasize.

---

### 4) The clinical covariates are small, but directionally plausible

* `p_diabetes` is mildly positive (0.163)
* `risk score` is mildly negative (-0.110)
* `CKD/COPD/cancer6` are mildly negative

Don’t over-interpret sign here. These features are provider-level aggregates and could correlate with lots of things. The right use is: 
> “Case-mix proxies contribute modestly relative to scale and specialty.”

## Checking the correlation between service volume and beneficiary count

Notice that `log_total_benes_base` has a coefficient of `0.011668`, which essentially means it has not effect on the odds of being in `cluster_1`. 

On the other hand, `log_total_services_base` has the third strongest positive coefficient indicating that higher the service volume the more likely to be in `cluster_1`. More specifically, for every 1 standard deviation increase in log of total volume, the odds of being in `cluster_1` goes up by `exp(0.737371)` which is equal to `2.090432536860409`. This is strong! 

Let's check the correlation between `log_total_benes_base` and `log_total_services_base`. 

In [ ]:
clf_df[clf_df["cluster_label_v1"].isin(["cluster_1", "cluster_0"])]["log_total_services_base"].corr(clf_df[clf_df["cluster_label_v1"].isin(["cluster_1", "cluster_0"])]["log_total_benes_base"])

# CLF.2.b) Full model (STATE included) + service volume only

In [ ]:
# ============================================================
# CLF.2.b) Full model (STATE included) + service volume only
# Minimal diff vs CLF.2.a:
#   - USE_SRVC_VOL instead of USE_SCALE
#   - keep state in CAT_FEATS
#   - drop log_total_benes_base (redundant w services)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

# Base features (no-leak)
NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type", "state"]

# Optional service-volume feature (toggle)
USE_SRVC_VOL = True
if USE_SRVC_VOL:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base"]

# Build X/y
X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

# Model
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

# Predict
p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("CLF.2.b (FULL): Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SRVC_VOL =", USE_SRVC_VOL)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

# ---- Interpret coefficients (log-odds) ----
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs}).sort_values("coef_log_odds", ascending=False)

print("\nTop positive drivers (higher => more likely cluster_1):")
display(coef_df.head(25))

print("\nTop negative drivers (lower => less likely cluster_1):")
display(coef_df.tail(25).iloc[::-1])

#### Checking the full `coef_df` dataframe output

In [ ]:
with pd.option_context("display.max_rows", None):
    display(coef_df)

# CLF.2.c) No-state model (DROP state) + service volume only

In [ ]:
# ============================================================
# CLF.2.c) No-state model (DROP state) + service volume only
# Goal:
#   - quantify how much state dummies are "carrying"
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type"]   # <-- state removed

USE_SRVC_VOL = True
if USE_SRVC_VOL:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base"]

X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("CLF.2.c (NO STATE): Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SRVC_VOL =", USE_SRVC_VOL)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs}).sort_values("coef_log_odds", ascending=False)

print("\nTop positive drivers:")
display(coef_df.head(25))

print("\nTop negative drivers:")
display(coef_df.tail(25).iloc[::-1])

#### Checking the full `coef_df` dataframe output

In [ ]:
with pd.option_context("display.max_rows", None):
    display(coef_df)

# CLF.2.d) Region-only model (region instead of state) + service volume only

In [ ]:
# ============================================================
# CLF.2.d) Region-only model (region instead of state) + service volume only
# Notes:
#   - Adds a simple Census-style region mapping.
#   - Drops state, adds region.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

# --- Region mapping (Census Bureau-style) ---
NORTHEAST = {"CT","ME","MA","NH","RI","VT","NJ","NY","PA"}
MIDWEST   = {"IL","IN","MI","OH","WI","IA","KS","MN","MO","NE","ND","SD"}
SOUTH     = {"DE","FL","GA","MD","NC","SC","VA","DC","WV","AL","KY","MS","TN","AR","LA","OK","TX"}
WEST      = {"AZ","CO","ID","MT","NV","NM","UT","WY","AK","CA","HI","OR","WA"}

def _state_to_region(s: str) -> str:
    s = str(s)
    if s in NORTHEAST: return "Northeast"
    if s in MIDWEST:   return "Midwest"
    if s in SOUTH:     return "South"
    if s in WEST:      return "West"
    return "Other"  # PR, VI, GU, XX, etc.

clf_tmp = clf_df.copy()
clf_tmp["region"] = clf_tmp["state"].map(_state_to_region).astype(str)

NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type", "region"]   # <-- region replaces state

USE_SRVC_VOL = True
if USE_SRVC_VOL:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base"]

X = clf_tmp[NUM_FEATS + CAT_FEATS].copy()
y = clf_tmp[TARGET].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("CLF.2.d (REGION): Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SRVC_VOL =", USE_SRVC_VOL)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs}).sort_values("coef_log_odds", ascending=False)

print("\nTop positive drivers:")
display(coef_df.head(25))

print("\nTop negative drivers:")
display(coef_df.tail(25).iloc[::-1])

print("\nRegion counts in eligible set:")
display(clf_tmp["region"].value_counts(dropna=False))

#### Checking the full `coef_df` dataframe output

In [ ]:
with pd.option_context("display.max_rows", None):
    display(coef_df)

### 1) CLF.2.b (FULL: state included)

**What performance says**
* **ROC AUC:** 0.736, **PR AUC:** 0.459, **accuracy:** 0.659.
* Compared to our base rate (~24.4% `cluster_1`), PR AUC ~0.46 is meaningfully better than chance, so the covariates are genuinely informative.
* **Confusion matrix:** True `cluster_1` detected: 321/457 (recall = 0.702)
* But precision for `cluster_1` is 0.390 (lots of false positives: 503).
* **Interpretation:** This is a “screening” model. It’s good at catching `cluster_1` providers, but it will over-flag. That is fine for an explanation module.

**What coefficients say**
The top drivers are mostly:
* **State dummies** (AK, ID, MS, GA, WY, OK, etc.)
* **Service volume:** `log_total_services_base` is strongly positive (0.749).
* **Provider type:** Medical Oncology (0.628) and Hematology-Oncology (0.434) positive. Surgical Oncology very negative (-1.209).

**Interpretation of the big one (service volume):**
With standardized features, a coefficient of 0.75 means: a 1 standard deviation increase in `log_total_services_base` multiplies the odds of being `cluster_1` by exp(0.75) ≈ **2.1x**.
This matches what we saw in tiering. `cluster_1` providers are much larger on services, even after the eligibility gate. Bigger providers have more “opportunity” to accumulate robust tail events.

**Interpretation of state:**
State is acting as a coarse proxy for lots of unmodeled stuff (local practice patterns, mix of facilities, regional coding norms, state-level provider composition).
The state coefficients are not “causal.” Many are also based on small counts (for example WY), so they can swing.

**Bottom line for FULL:** best raw predictive performance, but hard to defend as “drivers” because state dominates and can be noisy.

---

### 2) CLF.2.c (NO STATE)

**What performance says**
* **ROC AUC:** 0.730, **PR AUC:** 0.443, **accuracy:** 0.651.
* So removing state causes:
    * ROC drops ~0.0067 (0.7362 → 0.7295)
    * PR drops ~0.0163 (0.4594 → 0.4431)
* **Interpretation:** State adds a bit, but it’s not the whole story. Most of the predictive power remains without state.

**What coefficients say (now much cleaner)**
* **Top positive drivers:**
    * `log_total_services_base` (0.743) still the strongest.
    * `provider_type` Medical Oncology (0.613), Hematology-Oncology (0.423), Radiation Oncology (0.229).
    * `p_diabetes` positive but small (0.141).
* **Negatives:**
    * `years_since_enumeration` (-0.201) and `ruca_bucket_Urban` (-0.225) are modestly negative.
    * Surgical Oncology is strongly negative (-1.212).

**Interpretation:**
Once state is gone, the model’s “story” becomes interpretable:
1. Size (services)
2. Specialty mix
3. Some weak case-mix / geography texture (diabetes, urban/rural)

**Bottom line for NO STATE:** slightly weaker than FULL, but much more defensible for a “what predicts watchlist membership” narrative.

---

### 3) CLF.2.d (REGION instead of state)

**What performance says**
* **ROC AUC:** 0.729, **PR AUC:** 0.437, **accuracy:** 0.654.
* This is basically the same as no-state, and a hair worse on PR.
* **Interpretation:** coarse region doesn’t recover the lift that state provides. That means the extra information in state is not simply “Northeast vs South.” It’s either finer-grain geography or state-level idiosyncrasies.

**What coefficients say**
* **Top positive:**
    * `log_total_services_base` (0.725) again.
    * `provider_type` Medical Oncology (0.651), Hematology-Oncology (0.464).
    * `region_Other` (0.408) which is basically our non-state entries (PR/VI/GU/XX). Small counts, so treat cautiously.
* **Negative:**
    * `region_Northeast` (-0.270) and `region_Midwest` (-0.195) are mildly negative relative to the baseline region category (the baseline is whichever one-hot category is dropped by the encoder).

**Interpretation:**
Region adds a little “direction,” but doesn’t really change the model compared to no-state.
The region effects are small compared to services and provider type.

**Bottom line for REGION:** not worth it unless we want a gentle geo control without state. It does not add lift beyond no-state here.

---

### 4) Cross-model takeaways (the “executive story”)

**A) Geography matters, but it is not the main explanation**
FULL is best (PR AUC 0.459), but NO STATE is close (0.443). That gap is real but modest. So we can confidently say:
> “Geography explains some variation, but most signal is tied to provider characteristics and scale.”

**B) Service volume is the most stable driver across all variants**
It is the top coefficient in all three models, around 0.72–0.75. This lines up with our tiering profile where `cluster_1` has vastly higher median services.
This is a key narrative:
> “`cluster_1` is partially a scale phenomenon: larger, high-confidence providers are more likely to accumulate robust tail events.”

**C) Specialty is a consistent second-order driver**
Medical Oncology and Hematology-Oncology consistently raise odds of `cluster_1` vs the baseline provider type.
Surgical Oncology consistently lowers odds strongly.
This is useful because it is interpretable and stable across model variants.

**D) Case-mix proxies add only mild incremental signal (so far)**
The `p_*` condition prevalence features have small coefficients. That does not mean they are unimportant. It means, in this setup, they do not sharply separate `cluster_1` vs `cluster_0` once we already include provider type and scale.

---

### 5) Which model should we keep for the “interpretability module”?

If our goal is an interview-friendly “drivers” slide:

**Keep CLF.2.c (NO STATE) as the primary explanation model.**
* It is close in performance to FULL.
* It avoids the awkward “state coefficients” story.
* The drivers become clean: volume + specialty + mild case-mix.

**Optionally, keep FULL as a footnote:**
> “Including state modestly improves performance, suggesting additional geographic effects.”

# Rerun of CLF.2.c to ensure the model and coefficients persist to the following cells

In [ ]:
# ============================================================
# CLF.2.c) No-state model (DROP state) + service volume only
# Goal:
#   - quantify how much state dummies are "carrying"
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type"]   # <-- state removed

USE_SRVC_VOL = True
if USE_SRVC_VOL:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base"]

X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("CLF.2.c (NO STATE): Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SRVC_VOL =", USE_SRVC_VOL)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()
coef_df = pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs}).sort_values("coef_log_odds", ascending=False)

print("\nTop positive drivers:")
display(coef_df.head(25))

print("\nTop negative drivers:")
display(coef_df.tail(25).iloc[::-1])

# Checking the references for the categorical variables

The “reference” category is whatever level gets dropped by the one-hot encoder. With scikit-learn’s `OneHotEncoder`, that only happens if we set `drop=...`. In our code we did not set drop, so by default it uses full one-hot for each categorical feature. 

#### A) Verify our current encoder has no reference

In [ ]:
# --- Inspect current OneHotEncoder behavior in our fitted pipe ---
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]

print("ohe.drop:", ohe.drop)  # should be None with our current code
for feat, cats in zip(CAT_FEATS, ohe.categories_):
    print(f"\n{feat} categories (n={len(cats)}):")
    print(list(cats)[:20], "..." if len(cats) > 20 else "")

#### B) Make baselines explicit: pick the baseline category explicitly (best for storytelling)

In [ ]:
# ============================================================
# CLF.2.c.1) No-state model + service volume only (WITH explicit baselines + odds ratios)
# Adds:
#   - Prints baseline categories actually used by OneHotEncoder
#   - Adds odds ratios (exp(coef)) for interpretability
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

TARGET = "is_cluster_1"

NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
]
CAT_FEATS = ["ruca_bucket", "provider_type"]   # <-- state removed

USE_SRVC_VOL = True
if USE_SRVC_VOL:
    NUM_FEATS = NUM_FEATS + ["log_total_services_base"]

X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# Explicit baselines (reference categories)
BASELINE = {
    "ruca_bucket": "Urban",
    "provider_type": "Hematology-Oncology",
}
drop_map = [BASELINE.get(feat, "first") for feat in CAT_FEATS]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=drop_map), CAT_FEATS),
    ],
    remainder="drop",
)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

p_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
cm = confusion_matrix(y_test, pred_test)

print("CLF.2.c.1 (NO STATE): Eligible-only classification (cluster_1 vs cluster_0)")
print("USE_SRVC_VOL =", USE_SRVC_VOL)
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_test, digits=3))

# ---- Interpret coefficients (log-odds) + odds ratios ----
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]

# Print baselines actually used by the encoder
print("\nBaselines (reference categories) used by OneHotEncoder:")
for feat, cats, dropped_idx in zip(CAT_FEATS, ohe.categories_, ohe.drop_idx_):
    base = cats[dropped_idx] if dropped_idx is not None else None
    print(f"  {feat}: {base}")

cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
feat_names = NUM_FEATS + cat_names

coefs = pipe.named_steps["model"].coef_.ravel()

coef_df = (
    pd.DataFrame({"feature": feat_names, "coef_log_odds": coefs})
      .assign(odds_ratio=lambda d: np.exp(d["coef_log_odds"]))
      .sort_values("coef_log_odds", ascending=False)
)

print("\nTop positive drivers (higher => more likely cluster_1):")
display(coef_df.head(25))

print("\nTop negative drivers (lower => less likely cluster_1):")
display(coef_df.tail(25).iloc[::-1])

# Optional: a compact odds-ratio view for numeric features (OR per 1 SD, since we scaled numerics)
num_or = coef_df[coef_df["feature"].isin(NUM_FEATS)].copy()
if len(num_or):
    num_or = num_or.sort_values("odds_ratio", ascending=False)
    print("\nNumeric features: odds ratio per +1 SD (StandardScaler):")
    display(num_or)

#### Let's confirm the baselines

In [ ]:
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
print("ohe.drop:", ohe.drop)
for feat, cats, dropped in zip(CAT_FEATS, ohe.categories_, ohe.drop_idx_):
    base = cats[dropped] if dropped is not None else None
    print(f"{feat} baseline:", base)

# Interpreting CLF.2.c.1 (No State, Explicit Baselines, Odds Ratios)

Here’s a thorough interpretation of **CLF.2.c.1**, and then an executive-style takeaway we can drop into slides.

## What CLF.2.c.1 is actually modeling
* **Goal:** Explain what provider characteristics are associated with being in `cluster_1` (Elevated anomaly burden) vs `cluster_0` (Typical cost behavior), **within the eligible set only** (7,488 providers).
* **Target prevalence:** ~24.4% `cluster_1`. So this is an imbalanced problem, but not extreme.

## Why our coefficients are now “clean”
We explicitly set baselines for the one-hot encoded categoricals:
* `ruca_bucket` baseline: **Urban**
* `provider_type` baseline: **Hematology-Oncology**

So every categorical coefficient now means:
> “Compared to the baseline category, holding everything else fixed, how do the odds of being `cluster_1` change?”

And every numeric coefficient now means:
> “For a +1 SD increase in this numeric feature (because StandardScaler), how do odds change?”

---

## Model quality, in plain terms

### Discrimination metrics
* **ROC AUC = 0.729**
  * The model ranks a random `cluster_1` provider above a random `cluster_0` provider about **73% of the time**.
  * Good for explanation, not perfect prediction. That’s fine for the “interpretability module” goal.
* **PR AUC = 0.443**
  * With a baseline prevalence of ~0.244, a PR AUC of ~0.443 is meaningful lift over random.
  * This metric matters here because it focuses on the positive class (`cluster_1`).

### Confusion matrix at 0.50 threshold
* **TN**=896 | **FP**=519
* **FN**=134 | **TP**=323

**Interpretation:**
* **Recall** for `cluster_1` = **0.707** (it catches ~71% of `cluster_1` providers).
* **Precision** for `cluster_1` = **0.384** (many flagged as `cluster_1` are actually `cluster_0`).

This is exactly what we’d expect from:
* `class_weight="balanced"` (pushes recall up for the minority class),
* a default 0.50 threshold (not tuned),
* and a model built for explanation rather than a production alerting system.

If we ever want it to behave more like a “watchlist classifier,” we’d tune the threshold to trade recall vs precision. But for interpretation, this is already useful.

---

## Interpreting the odds ratios (the core deliverable)

**Key point: Odds ratios are conditional and relative.** Every odds ratio is “all else equal” within this model.

### Biggest driver by far: service volume
`log_total_services_base`
* **coef** = 0.745
* **odds ratio** = 2.11

**Meaning:** A **+1 SD** increase in `log_total_services_base` multiplies the odds of being `cluster_1` by **~2.1x**.

**Why this makes sense in our project:**
* `cluster_1` is the “elevated anomaly burden” bucket.
* Bigger providers have more “opportunities” to generate robust tail events, even with rates included, because our robust event definition is based on tail membership + confidence gate + slice validity.
* We already saw in tiering that `cluster_1` has much higher median services than `cluster_0`. This model is quantifying that effect.

**An important narrative point:**
> “In this dataset, **size is a major contributor** to being in the elevated-anomaly cluster, even when we gate eligibility by minimum evidence.”

### RUCA bucket effects (relative to Urban baseline)
* `ruca_bucket_Suburban`: **OR 1.25**
* `ruca_bucket_Rural`: **OR 1.23**

**Interpretation:** Relative to Urban, Suburban and Rural providers have **~23–25% higher odds** of being `cluster_1`, holding other covariates fixed. These are modest effects compared to service volume.

**A nuance:** RUCA category can be acting as a proxy for practice patterns, referral networks, site-of-service mix, or coding/billing norms. Since we dropped state, RUCA is one of the remaining context variables.

### Provider type effects (relative to Hematology-Oncology baseline)
* `provider_type_Medical Oncology`: **OR 1.22** (Medical Oncology has ~22% higher odds than Hem/Onc, all else equal.)
* `provider_type_Radiation Oncology`: **OR 0.83** (Radiation Oncology has ~17% lower odds than Hem/Onc.)
* `provider_type_Gynecological Oncology`: **OR 0.49** (About half the odds vs Hem/Onc.)
* `provider_type_Surgical Oncology`: **OR 0.22** (Much lower odds vs Hem/Onc.)

**Important caution:** Some of these types likely have smaller sample sizes and different service mixes. The direction is still interpretable as “in this dataset, `cluster_1` is more associated with certain oncology provider types,” but we should avoid implying causal “quality” differences.

### Case-mix and risk proxies (all per +1 SD)
* `p_diabetes`: **OR 1.15** (More diabetes mix is associated with higher `cluster_1` odds, modestly.)
* `years_since_enumeration`: **OR 0.82** (Older NPIs have lower `cluster_1` odds, modestly.)
* `bene_avg_risk_score`: **OR 0.89** (Higher risk score is associated with lower `cluster_1` odds here—counterintuitive on first glance.)
* `p_cancer6`, `p_copd`, `p_ckd`: **OR in 0.86–0.90 range** (Slightly protective associations in this model.)

**How to read these without over-interpreting:**
These variables are “proxy covariates” aggregated from `eval_base`, not a true clinical case-mix adjustment. Some of them may be correlated with provider type, services, or coding patterns. Logistic regression coefficients reflect conditional relationships after accounting for the other predictors. It is normal for some clinical proxies to flip direction depending on what else is in the model.

If we want to sanity-check these, the next good move is: fit the same model without `log_total_services_base` and see which case-mix coefficients change most (size can absorb a lot).

---

## Why CLF.2.c.1 is the best “explanation” logistic model

1. **It removes the geography shortcut:** When state is included, state dummies dominate coefficients and can look like “the answer.” Dropping state forces the model to explain `cluster_1` using provider attributes and context that generalize better.
2. **It keeps one clear scale signal (and we made it non-redundant):** We confirmed `log_total_services_base` and `log_total_benes_base` are highly correlated (~0.83). Keeping only `log_total_services_base` gives a clean “size effect” without multicollinearity clutter.
3. **Coefficients are interpretable and stable-looking:** The top drivers are conceptually coherent (service volume, provider type, RUCA). This is exactly what we want for an interpretability module and interview narrative.
4. **It performs well enough for insight:** ROC AUC ~0.73 and PR AUC ~0.44 are strong enough to justify “the cluster is learnable from covariates,” without claiming production-grade prediction.

---

## Executive summary we can use (slide-ready)

* **Objective:** Explain what provider characteristics are associated with being in the “Elevated anomaly burden” cluster among eligible providers.
* **Approach:** Logistic regression on clustered providers only (n=7,488), excluding state to avoid geography dominating interpretation. Numeric predictors are standardized, categorical predictors use explicit reference categories.
* **Performance:** ROC AUC ≈ **0.73**, PR AUC ≈ **0.44**. Good explanatory power for an insight model.
* **Primary drivers (odds ratios):**
  * **Service volume** (`log_total_services_base`): ~2.1x higher odds per +1 SD
  * **RUCA context** (vs Urban): Suburban ~1.25x, Rural ~1.23x
  * **Provider type** (vs Hematology-Oncology): Medical Oncology ~1.22x
* **Interpretation:** The elevated-anomaly cluster is strongly associated with provider scale, and secondarily with provider type and geography context (RUCA), consistent with the tiering result that `cluster_1` providers are much larger and accumulate repeat tail events.
* **Use:** Not for decision-making. For explaining what correlates with “monitoring tier” membership in this dataset.

# CLF.2.c.1.1) Bootstrap stability for Logistic (NO STATE, explicit baselines)

In [ ]:
# ============================================================
# CLF.2.c.1.1) Bootstrap stability for Logistic (NO STATE, explicit baselines)
# Reports:
#   A) Top-k stability: how often each feature appears in top-k by |coef|
#   B) Odds-ratio distribution per feature across resamples (median/p05/p95)
# Notes:
#   - Uses the SAME feature set, preprocessing, baselines, and model as CLF.2.c.1
#   - Bootstraps the eligible provider dataset with replacement
#   - Coefficients are in log-odds; odds_ratio = exp(coef)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# -----------------------------
# Preconditions
# -----------------------------
if "clf_df" not in globals():
    raise NameError("Missing clf_df. Run CLF.1 first.")
if "TARGET" in globals():
    TARGET_COL = TARGET
else:
    TARGET_COL = "is_cluster_1"

# -----------------------------
# Settings (tune as needed)
# -----------------------------
B = 200          # bootstrap resamples
TOP_K = 10       # track stability in top-k by abs(coef)
RANDOM_STATE = 7

# Same features as CLF.2.c.1
NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
    "log_total_services_base",
]
CAT_FEATS = ["ruca_bucket", "provider_type"]

# Explicit baselines as in CLF.2.c.1
BASELINE = {
    "ruca_bucket": "Urban",
    "provider_type": "Hematology-Oncology",
}

# -----------------------------
# Build X/y once
# -----------------------------
need_cols = NUM_FEATS + CAT_FEATS + [TARGET_COL]
missing = [c for c in need_cols if c not in clf_df.columns]
if missing:
    raise KeyError(f"clf_df missing required cols: {missing}")

X_all = clf_df[NUM_FEATS + CAT_FEATS].copy()
y_all = clf_df[TARGET_COL].astype(int).to_numpy()

# -----------------------------
# Preprocess + model factory (must rebuild per resample)
# -----------------------------
drop_map = []
for feat in CAT_FEATS:
    drop_map.append(BASELINE.get(feat, "first"))  # fallback

def make_pipe():
    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUM_FEATS),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop=drop_map), CAT_FEATS),
        ],
        remainder="drop",
    )
    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
    )
    return Pipeline(steps=[("preprocess", preprocess), ("model", clf)])

# -----------------------------
# Feature-name helper (consistent order with coefficient vector)
# -----------------------------
def get_feature_names(fit_pipe: Pipeline) -> list:
    ohe = fit_pipe.named_steps["preprocess"].named_transformers_["cat"]
    cat_names = list(ohe.get_feature_names_out(CAT_FEATS))
    return NUM_FEATS + cat_names

# -----------------------------
# Bootstrap loop
# -----------------------------
rng = np.random.default_rng(RANDOM_STATE)
n = len(X_all)

rows = []
topk_counts = {}  # feature -> count in top-k

for b in range(B):
    idx = rng.integers(0, n, size=n)  # bootstrap sample with replacement
    Xb = X_all.iloc[idx].copy()
    yb = y_all[idx].copy()

    pipe_b = make_pipe()
    pipe_b.fit(Xb, yb)

    feat_names = get_feature_names(pipe_b)
    coefs = pipe_b.named_steps["model"].coef_.ravel()

    # Store coefficients + odds ratios
    for f, c in zip(feat_names, coefs):
        rows.append({"boot": b, "feature": f, "coef_log_odds": float(c), "odds_ratio": float(np.exp(c))})

    # Top-k by absolute coefficient
    abs_idx = np.argsort(np.abs(coefs))[::-1][:TOP_K]
    for j in abs_idx:
        f = feat_names[j]
        topk_counts[f] = topk_counts.get(f, 0) + 1

boot_df = pd.DataFrame(rows)

# -----------------------------
# A) Top-k stability table
# -----------------------------
stability = (
    pd.DataFrame({"feature": list(topk_counts.keys()), "topk_hits": list(topk_counts.values())})
    .assign(topk_share=lambda d: d["topk_hits"] / B)
    .sort_values(["topk_hits", "feature"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Bootstrap stability: B={B}, TOP_K={TOP_K}")
display(stability.head(30))

# -----------------------------
# B) Odds-ratio distribution table (median/p05/p95)
# -----------------------------
def _q05(x): return float(np.nanquantile(x, 0.05))
def _q95(x): return float(np.nanquantile(x, 0.95))

or_dist = (
    boot_df.groupby("feature", dropna=False)["odds_ratio"]
    .agg(or_median="median", or_p05=_q05, or_p95=_q95)
    .reset_index()
)

# Attach top-k share (0 if never top-k)
or_dist = or_dist.merge(stability[["feature", "topk_share"]], on="feature", how="left")
or_dist["topk_share"] = or_dist["topk_share"].fillna(0.0)

# Nice ordering: most stable first
or_dist = or_dist.sort_values(["topk_share", "or_median"], ascending=[False, False]).reset_index(drop=True)

print("\nOdds-ratio distribution across bootstraps (odds ratio per +1 SD for numeric, vs baseline for cats):")
display(or_dist.head(40))

# Optional: focused view for “headline drivers”
HEADLINE = [
    "log_total_services_base",
    "provider_type_Medical Oncology",
    "provider_type_Radiation Oncology",
    "provider_type_Gynecological Oncology",
    "provider_type_Surgical Oncology",
    "ruca_bucket_Suburban",
    "ruca_bucket_Rural",
]
headline_rows = or_dist[or_dist["feature"].isin(HEADLINE)].copy()
if len(headline_rows):
    print("\nHeadline drivers (for exec narrative):")
    display(headline_rows)

# Baseline reminder
print("\nBaselines (reference categories):")
for k, v in BASELINE.items():
    print(f"  {k}: {v}")

# XGBoost Classification

# CLF.3) XGBoost (no-state primary) on same feature set as CLF.2.c.1


In [ ]:
# ============================================================
# CLF.3) XGBoost (no-state primary) on same feature set as CLF.2.c.1
# Features:
#   NUM: case-mix proxies + risk + years_since_enumeration + log_total_services_base
#   CAT: ruca_bucket + provider_type
# Evaluates:
#   ROC AUC, PR AUC, Brier score, calibration curve
# Produces:
#   - xgb_pipe (trained pipeline)
#   - X_train/X_test/y_train/y_test (shared split for comparisons)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, classification_report
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

from xgboost import XGBClassifier

# -----------------------------
# Preconditions
# -----------------------------
if "clf_df" not in globals():
    raise NameError("Missing clf_df. Run CLF.1 first.")

TARGET = "is_cluster_1"

NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
    "log_total_services_base",
]
CAT_FEATS = ["ruca_bucket", "provider_type"]  # no-state

need = NUM_FEATS + CAT_FEATS + [TARGET]
missing = [c for c in need if c not in clf_df.columns]
if missing:
    raise KeyError(f"clf_df missing required columns: {missing}")

X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET].astype(int).to_numpy()

# Shared split for all CLF.3.* comparisons
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# -----------------------------
# Preprocess: OHE for categoricals, pass-through numerics
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

# -----------------------------
# XGBoost classifier
# -----------------------------
pos_rate = float(y_train.mean())
scale_pos_weight = float((1 - pos_rate) / pos_rate) if pos_rate > 0 else 1.0

xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=7,
    n_jobs=-1,
)

xgb_pipe = Pipeline(steps=[("preprocess", preprocess), ("model", xgb)])
xgb_pipe.fit(X_train, y_train)

# -----------------------------
# Predict + metrics
# -----------------------------
p_test = xgb_pipe.predict_proba(X_test)[:, 1]
pred_05 = (p_test >= 0.50).astype(int)

roc = float(roc_auc_score(y_test, p_test))
pr = float(average_precision_score(y_test, p_test))
brier = float(brier_score_loss(y_test, p_test))
cm = confusion_matrix(y_test, pred_05)

print("CLF.3 (XGBoost, NO STATE): Eligible-only classification (cluster_1 vs cluster_0)")
print("ROC AUC:", roc)
print("PR AUC:", pr)
print("Brier score (lower better):", brier)
print("\nConfusion @0.50 threshold:\n", cm)
print("\nClassification report:\n", classification_report(y_test, pred_05, digits=3))

# -----------------------------
# Calibration curve
# -----------------------------
prob_true, prob_pred = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111)
ax.plot(prob_pred, prob_true, marker="o")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("Calibration curve (XGBoost, no-state)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
plt.tight_layout()
plt.show()

# CLF.3.A) Compare XGBoost vs Logistic baseline on SAME split

In [ ]:
# ============================================================
# CLF.3.A) Compare XGBoost vs Logistic baseline on SAME split
# Uses:
#   - X_train/X_test/y_train/y_test from CLF.3
# Trains:
#   - Logistic Regression baseline (balanced) with same features (no-state)
# Reports:
#   - ROC AUC, PR AUC, Brier for both
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# Preconditions
for v in ["X_train", "X_test", "y_train", "y_test", "NUM_FEATS", "CAT_FEATS", "xgb_pipe"]:
    if v not in globals():
        raise NameError(f"Missing {v}. Run CLF.3 first.")

# Logistic pipeline (no-state), keep numeric scaling for odds-ratio interpretability
log_pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
    ],
    remainder="drop",
)

log_clf = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
log_pipe = Pipeline(steps=[("preprocess", log_pre), ("model", log_clf)])
log_pipe.fit(X_train, y_train)

# Predict probabilities
p_xgb = xgb_pipe.predict_proba(X_test)[:, 1]
p_log = log_pipe.predict_proba(X_test)[:, 1]

rows = []
for name, p in [("XGBoost", p_xgb), ("Logistic", p_log)]:
    rows.append({
        "model": name,
        "roc_auc": float(roc_auc_score(y_test, p)),
        "pr_auc": float(average_precision_score(y_test, p)),
        "brier": float(brier_score_loss(y_test, p)),
    })

compare = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
display(compare)

# CLF.4) Linear SVM sanity check (no-state) vs Logistic (same split)

In [ ]:
# ============================================================
# CLF.3.SANITY) Precision@k (watchlist triage) for XGBoost vs Logistic
# Reports:
#   - precision@top5%, top10%, top20%
#   - recall@top5%, top10%, top20% (optional but useful)
# ============================================================

import numpy as np
import pandas as pd

# Preconditions
for v in ["y_test", "p_xgb", "p_log"]:
    if v not in globals():
        raise NameError(f"Missing {v}. Run CLF.3 and CLF.3.A first.")

def precision_recall_at_k(y_true: np.ndarray, p: np.ndarray, frac: float) -> dict:
    n = len(y_true)
    k = max(1, int(np.floor(frac * n)))
    idx = np.argsort(p)[::-1][:k]
    y_sel = y_true[idx]
    prec = float(y_sel.mean())
    rec = float(y_sel.sum() / max(1, y_true.sum()))
    return {"k_frac": frac, "k": k, "precision": prec, "recall": rec}

fracs = [0.05, 0.10, 0.20]

rows = []
for frac in fracs:
    r1 = precision_recall_at_k(y_test, p_xgb, frac)
    r1["model"] = "XGBoost"
    rows.append(r1)

    r2 = precision_recall_at_k(y_test, p_log, frac)
    r2["model"] = "Logistic"
    rows.append(r2)

triage = pd.DataFrame(rows).sort_values(["k_frac", "model"])
display(triage)

# SVM for a quick comparison

In [ ]:
# ============================================================
# CLF.4) Linear SVM sanity check (no-state) vs Logistic (same split)
# Goal:
#   - Run ONE linear SVM as a quick sanity benchmark (no tuning spiral)
#   - Compare against your locked Logistic baseline using the SAME split
# Outputs:
#   - ROC AUC, PR AUC
#   - Precision@k (5%, 10%, 20%) to match watchlist triage
# Notes:
#   - Uses SAME feature set + baselines as CLF.2.c.1.1
#   - LinearSVC has no native probabilities, so we use CalibratedClassifierCV
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# -----------------------------
# Preconditions
# -----------------------------
if "clf_df" not in globals():
    raise NameError("Missing clf_df. Run CLF.1 first.")

TARGET_COL = "is_cluster_1"

# Same as CLF.2.c.1.1
NUM_FEATS = [
    "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
    "bene_avg_risk_score","years_since_enumeration",
    "log_total_services_base",
]
CAT_FEATS = ["ruca_bucket", "provider_type"]

BASELINE = {
    "ruca_bucket": "Urban",
    "provider_type": "Hematology-Oncology",
}

need_cols = NUM_FEATS + CAT_FEATS + [TARGET_COL]
missing = [c for c in need_cols if c not in clf_df.columns]
if missing:
    raise KeyError(f"clf_df missing required cols: {missing}")

X = clf_df[NUM_FEATS + CAT_FEATS].copy()
y = clf_df[TARGET_COL].astype(int).to_numpy()

# -----------------------------
# Same split as your baseline (fixed seed + stratify)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# -----------------------------
# Shared preprocessing (explicit baselines)
# -----------------------------
drop_map = []
for feat in CAT_FEATS:
    drop_map.append(BASELINE.get(feat, "first"))

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=drop_map), CAT_FEATS),
    ],
    remainder="drop",
)

# -----------------------------
# Model 1: Logistic (baseline reference)
# -----------------------------
log_clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)

log_pipe = Pipeline(steps=[("preprocess", preprocess), ("model", log_clf)])
log_pipe.fit(X_train, y_train)

p_log = log_pipe.predict_proba(X_test)[:, 1]

# -----------------------------
# Model 2: Linear SVM + calibration for probabilities
# -----------------------------
# LinearSVC gives decision_function; calibration converts to probabilities.
svm_base = Pipeline(steps=[
    ("preprocess", preprocess),
    ("svm", LinearSVC(class_weight="balanced", random_state=7)),
])

# Calibrate on training only (internal CV on training)
svm_cal = CalibratedClassifierCV(
    estimator=svm_base,
    method="sigmoid",   # Platt scaling (simple, stable)
    cv=5,
)
svm_cal.fit(X_train, y_train)

p_svm = svm_cal.predict_proba(X_test)[:, 1]

# -----------------------------
# Metrics helper
# -----------------------------
def _metrics(y_true, p):
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
    }

def _precision_at_k(y_true, p, k_frac):
    n = len(y_true)
    k = max(1, int(round(k_frac * n)))
    idx = np.argsort(p)[::-1][:k]
    prec = float(y_true[idx].mean())
    rec = float(y_true[idx].sum() / max(1, y_true.sum()))
    return {"k_frac": float(k_frac), "k": int(k), "precision": prec, "recall": rec}

rows = []
rows.append({"model": "Logistic", **_metrics(y_test, p_log)})
rows.append({"model": "LinearSVM_calibrated", **_metrics(y_test, p_svm)})

cmp = pd.DataFrame(rows).sort_values("model")
print("CLF.4 comparison (same split, no-state):")
display(cmp)

# Precision@k (watchlist triage)
k_fracs = [0.05, 0.10, 0.20]
pk_rows = []
for kf in k_fracs:
    pk_rows.append({**_precision_at_k(y_test, p_log, kf), "model": "Logistic"})
    pk_rows.append({**_precision_at_k(y_test, p_svm, kf), "model": "LinearSVM_calibrated"})

pk = pd.DataFrame(pk_rows).sort_values(["k_frac", "model"])
print("\nPrecision@k (higher precision is better for a watchlist):")
display(pk)

# CLF.DECISION) Lock final explanation model

In [ ]:
# ============================================================
# CLF.DECISION) Lock final explanation model
# Goal:
#   - One cell that records the decision in notebook state for clarity
# ============================================================

FINAL_EXPLANATION_MODEL = "Logistic (CLF.2.c.1.1, no-state, explicit baselines)"
FINAL_EXPLANATION_FEATURES = {
    "numeric": [
        "p_cancer6","p_diabetes","p_ckd","p_copd","p_htn",
        "bene_avg_risk_score","years_since_enumeration",
        "log_total_services_base",
    ],
    "categorical": ["ruca_bucket", "provider_type"],
    "excluded": ["state"],  # intentionally excluded for transferability
    "baselines": {
        "ruca_bucket": "Urban",
        "provider_type": "Hematology-Oncology",
    },
}

print("✅ Final explanation model locked:")
print("  ", FINAL_EXPLANATION_MODEL)
print("\nFeature set:")
for k, v in FINAL_EXPLANATION_FEATURES.items():
    print(f"  {k}: {v}")

# CLF.FREEZE) Freeze final explanation model

In [ ]:
# ============================================================
# CLF.FREEZE) Freeze final explanation model
# Writes:
#   - final_explainer_logistic__{ts}.joblib
#   - params__{ts}.json + params__{ts}.md + manifest__{ts}.json
# Notes:
#   - Expects the fitted pipeline from CLF.2.c.1 to exist as `pipe`
#   - Uses explicit baselines and no-state features
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd

# ---- Preconditions ----
if "pipe" not in globals():
    raise NameError("Missing `pipe` (fitted Logistic pipeline). Run CLF.2.c.1 and keep the fitted pipeline named `pipe`.")
if "BASELINE" not in globals():
    raise NameError("Missing BASELINE dict. Run CLF.2.c.1.")
if "NUM_FEATS" not in globals() or "CAT_FEATS" not in globals():
    raise NameError("Missing NUM_FEATS/CAT_FEATS. Run CLF.2.c.1.")

ts = pd.Timestamp.now(tz="America/New_York").strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("artifacts/provider_classification") / f"run_{ts}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUT_DIR / f"final_explainer_logistic__{ts}.joblib"
PARAMS_JSON = OUT_DIR / f"params__{ts}.json"
MANIFEST_JSON = OUT_DIR / f"manifest__{ts}.json"
PARAMS_MD = OUT_DIR / f"params__{ts}.md"

joblib.dump(pipe, MODEL_PATH)

PARAMS = {
    "timestamp_et": ts,
    "outputs_dir": str(OUT_DIR),
    "final_explanation_model": "LogisticRegression (balanced) in Pipeline(preprocess -> model)",
    "target": "is_cluster_1",
    "features": {
        "numeric": list(NUM_FEATS),
        "categorical": list(CAT_FEATS),
        "excluded": ["state"],
        "baselines": dict(BASELINE),
    },
    "preprocess": {
        "numeric_scaler": "StandardScaler",
        "categorical_encoder": "OneHotEncoder(handle_unknown='ignore', drop=baselines)",
    },
    "artifact": {
        "model_joblib": str(MODEL_PATH),
    },
}

with open(PARAMS_JSON, "w") as f:
    json.dump(PARAMS, f, indent=2)

with open(MANIFEST_JSON, "w") as f:
    json.dump(
        {"timestamp_et": ts, "outputs_dir": str(OUT_DIR),
         "artifacts": [{"name": "final_explainer_logistic", "joblib": str(MODEL_PATH)}]},
        f, indent=2
    )

md = []
md.append(f"# Provider classification explainer (final Logistic) — {ts} ET\n\n")
md.append(f"Output directory: `{OUT_DIR}`\n\n")
md.append("## Artifact\n")
md.append(f"- Joblib: `{MODEL_PATH}`\n\n")
md.append("## Parameters\n")
md.append("```json\n" + json.dumps(PARAMS, indent=2) + "\n```\n")
PARAMS_MD.write_text("".join(md))

print("✅ Frozen final explanation model")
print(" -", MODEL_PATH)
print(" -", PARAMS_JSON)
print(" -", PARAMS_MD)
print(" -", MANIFEST_JSON)